## 处理Backbone数据

In [2]:
## Read excel
import pandas as pd 
sheet_names = ['核实', '告知', '三方', '转告', '资金困难', '信息问题', '承诺还款', '非明确承诺还款', '无法沟通', '特殊情况', '用卡及征信', '年费及可用额度', '投诉处理', '放时间应对', '要求领导回电应对', '要求停催应对', '联系三方应对', '转人工及挂机', '衔接施压话术']
columns_names = ['uid', 'human(客户)', 'assistant(专员)', 'conditions(条件)', 'parent(继承)', 'flexible_stop(可选不继承)', 'repeat(次数)', '是否再见']

df_dict = {}

for sheet_name in sheet_names:
    df = pd.read_excel("对话模块分类 - 洋钱罐 - 1111.xlsx", sheet_name=sheet_name)
    
    # 筛选conditions(条件)列精确等于"S1|S2"的行
    df_filtered = df[df['conditions(条件)'] == 'S1|S2']

    # 筛选掉话术里带有随机金额和随机时间的行
    df_filtered = df_filtered[~df_filtered['human(客户)'].str.contains('<随机金额>|<随机时间>', na=False)]
    
    df_dict[sheet_name] = df_filtered

len(sheet_names)

19

In [3]:
df_dict['放时间应对']

,uid,human(客户),assistant(专员),conditions(条件),parent(继承),flexible_stop(可选不继承),repeat(次数),是否再见
0,1,能不能晚点啊。/可不可以晚点啊。/我晚点还可以吧。/晚一点再说吧。,晚点是什么时间呢，还款是有时间要求的，不是你想什么时候还就什么时候还的。/要晚到什么时候呢？,S1|S2,0,0,1/2,0
1,2,能不能给我点时间呢？/再给我点时间吧。/再给我点时间不行吗？/不能往后延延吗？,不是不给你时间，已经给了你很长时间了。/已经给了你足够的时间，你这段时间没有去想办法吗？,S1|S2,0,0,1/2,0
2,3,下班回去还。/我现在没时间，下班再还。/下班吧，下班吧，我下班就还进去。/我在你下班之前还进去吧。,现在你这个事情比较紧急，没办法等你下班再处理。/现在还款也很方便，不会耽误你很长时间，建议你...,S1|S2,0,0,1/2,0
3,4,过两天还。/过两天吧。/现在不方便，过两天还。,给不了你更长的时间了，你现在已经逾期了，现在电话通知到你，是需要你按照公司要求还款。/如果能...,S1|S2,0,0,1/2,0
4,5,今天不行，等我发工资。/<银行查账时间>不行，我只能等发工资才能还。/等我发工资就还。/但是...,那你什么时候发工资呢，已经给了你这么长时间了，你也没有按要求还款啊。/那您之前发的工资怎么没...,S1|S2,0,0,1/2,0
7,8,过段时间还。/过段时间吧。/过段时间我还进去。/等过段时间再说吧,等不了你那么长时间，毕竟现在已经逾期了，打电话来就是提醒你还款的，如果能给你时间，今天就不会...,S1|S2,0,0,1/2,0
8,9,<银行查账时间>不一定，不要规定时间行不行？我今天肯定处理掉。/我今天给你还进去不就行了。/...,我相信你今天肯定能处理到位，但是<银行查账时间>是公司规定的查账时间，人工无法干涉，需要你克...,S1|S2,0,0,1/2,0
9,10,我现在在上班呢，怎么给你还，下班吧，我下班就还进去。/我现在上班没时间还，等我下班再说。/工...,只要你有钱，现在还款是非常方便的，微信支付宝耽误不了你两分钟的时间，你抽个两分钟时间还进去就...,S1|S2,0,0,1/2,0
10,11,手里现在没那么多钱，过两天吧，过两天就可以了。/过两天我就还，现在钱不够。,你现在已经逾期<最大逾期天数>天了，没办法再等你这么长时间了。/你现在都已经逾期了，没有办法...,S1|S2,0,0,1/2,0
11,12,你这是什么意思，我又不是不还，说了过两天还就过两天还，我还能跑了不成。/我没说我不还，我过两...,我知道你肯定会还的，但是公司的还款时间也是有要求的，你现在是已经逾期了，没办法再给你时间了。...,S1|S2,0,0,1/2,0


## 按概率矩阵生成路径

####
1. 暂时不处理：没有带衔接话术的模块不能接非明确承诺还款
2. conditions不处理
3. 继承不算次数

In [4]:
modules = [
    '核实',
    '三方',
    '转告',
    '告知',
    '承诺还款',
    '非明确承诺还款',
    '资金困难',
    '信息问题',
    '用卡及征信',
    '无法沟通',
    '放时间应对',
    '特殊情况',
    '年费及可用额度',
    '投诉处理',
    '联系三方应对',
    '要求停催应对',
    '要求领导回电应对',
    '转人工及挂机'
]
possible_times = [4,2,2,2,2,3,4,4,3,3,3,3,2,3,2,3,2,3]

In [5]:
max_counts = dict(zip(modules, possible_times))
max_counts

{'核实': 4,
 '三方': 2,
 '转告': 2,
 '告知': 2,
 '承诺还款': 2,
 '非明确承诺还款': 3,
 '资金困难': 4,
 '信息问题': 4,
 '用卡及征信': 3,
 '无法沟通': 3,
 '放时间应对': 3,
 '特殊情况': 3,
 '年费及可用额度': 2,
 '投诉处理': 3,
 '联系三方应对': 2,
 '要求停催应对': 3,
 '要求领导回电应对': 2,
 '转人工及挂机': 3}

In [6]:
import pandas as pd

file_path = "prob.xlsx"
df = pd.read_excel(file_path, header=None)

print(df.sum(axis=1))

df.columns = modules
df.index = modules

print(df)

0     100
1     100
2     100
3     100
4     100
5     100
6     100
7     100
8     100
9     100
10    100
11    100
12    100
13    100
14    100
15    100
16    100
17    100
dtype: int64
          核实  三方   转告  告知  承诺还款  非明确承诺还款  资金困难  信息问题  用卡及征信  无法沟通  放时间应对  \
核实        15   5    0  80     0        0     0     0      0     0      0   
三方         0  50   50   0     0        0     0     0      0     0      0   
转告         0   0  100   0     0        0     0     0      0     0      0   
告知         0   0    0   0     5        8    20    15     12     2     20   
承诺还款       0   0    0   0    30        5     1    30     25     1      1   
非明确承诺还款    0   0    0   0    15       25     8    15      8     2     12   
资金困难       0   0    0   0     5        5    30    15      8     1     25   
信息问题       0   0    0   0     5        5    20    30     15     1     15   
用卡及征信      0   0    0   0     5        5    23    15     20     1     18   
无法沟通       0   0    0   0     3        3    16 

In [7]:
import numpy as np

num_paths = 40000      # 生成路径数量
# max_length = 8      # 每条路径最多步数(先不用)
special_nodes = {'特殊情况', '信息问题', '用卡及征信'}

all_paths = []

for _ in range(num_paths):
    path = ["核实"]
    node_counts = {node: 0 for node in modules}
    node_counts["核实"] = 1
    banned_nodes = set()
    
    current_node = "核实"
    
    # Random walk
    while True:
        prob_dist = df.loc[current_node].copy() / 100

        available_nodes = [n for n in modules if n not in banned_nodes]
        if not available_nodes:
            break

        prob_dist[~prob_dist.index.isin(available_nodes)] = 0
        if prob_dist.sum() == 0:
            break
        prob_dist /= prob_dist.sum()

        next_node = np.random.choice(modules, p=prob_dist)

        path.append(next_node)
        node_counts[next_node] += 1
        
        # Check
        if node_counts[next_node] >= max_counts[next_node]:
            if next_node in special_nodes:
                banned_nodes.add(next_node)
            else:
                break
        
        current_node = next_node

    all_paths.append(path)

In [8]:
all_paths[:5]

[['核实', '告知', '要求停催应对', '投诉处理', '投诉处理', '特殊情况', '投诉处理'],
 ['核实',
  '核实',
  '核实',
  '告知',
  '资金困难',
  '用卡及征信',
  '资金困难',
  '非明确承诺还款',
  '资金困难',
  '放时间应对',
  '非明确承诺还款',
  '非明确承诺还款'],
 ['核实',
  '告知',
  '资金困难',
  '放时间应对',
  '放时间应对',
  '资金困难',
  '非明确承诺还款',
  '投诉处理',
  '联系三方应对',
  '联系三方应对'],
 ['核实', '三方', '转告', '转告'],
 ['核实', '告知', '资金困难', '资金困难', '资金困难', '资金困难']]

## 根据路径抽样，每个sheet采样

In [9]:
# 处理衔接施压话术
## 直接按repeat 1,2,3各抽一个，不考虑继承，然后插入到路径中去

def added_utterance():
    dialog_parent = []

    df_node = df_dict['衔接施压话术']
    for repeat in [1, 2, 3]:
        df_node_repeat = df_node[
            df_node['repeat(次数)'].apply(
                lambda x: str(repeat) in str(x).split('/') if pd.notna(x) else False
            )
        ]

        # 只考虑parent=0的行
        df_node_repeat_sub = df_node_repeat[df_node_repeat['parent(继承)']==0]

        sampled_row = df_node_repeat_sub.sample(n=1).iloc[0]

        assistant_utterances = sampled_row['assistant(专员)'].split('/')
        assistant_utterance = np.random.choice(assistant_utterances).strip()

        dialog_parent.append(assistant_utterance)
    return dialog_parent

print(added_utterance())

['你现在已经严重违约，公司有权利让你一次性还全款，但是考虑你资金问题，现在只需要你在<银行查账时间>前归还<剩余总待还>，应该没问题吧？', '逾期会影响征信，你应该知道吧？', '<银行查账时间>是平台系统统一查账时间，届时如果查到空账少账，不排除可能会采取面对面的沟通方式来跟您本人重新核实贷款时的信息，一旦流程开展，这个影响你有考虑过吗？']


In [10]:
# for 每个路径：
    # for 每个节点：
        # 判断该节点在路径中已出现的次数（repeat）
            # 抽行，行内根据'/'抽话术，同时判断有没有继承（往上或者往下），继承就加进去
            # 判断该行有没有再见，暂时先不管！！！！！！！！！！！！！！！！！！！！！！！！！！！！
            # 判断是否到了最大次数限制，暂时没管！！！！！！！！！！！！！！！！！！！！！！！！！！
            # 处理衔接施压话术---没有考虑继承！！！！！！！！！！！！！！！！！！！！！！！！！！！
    # 得到对话后，最后替换标签

dialogs = []
# for path in all_paths[:2]:

for path in all_paths:
    dialog = []
    cnt_node = {}
    # 生成三个候补的衔接施压话术
    insert_nodes = ['资金困难', '用卡及征信', '放时间应对']
    dialog_parent = added_utterance()
    parent_index = 0
    for node in path:
        # 判断当前节点在路径中已出现的次数
        if node not in cnt_node:
            cnt_node[node] = 0
        cnt_node[node] += 1
        repeat = cnt_node[node]

        df_node = df_dict[node]
        # df_node['repeat(次数)'] 可能包含'/'
        df_node_repeat = df_node[
            df_node['repeat(次数)'].apply(
                lambda x: str(repeat) in str(x).split('/') if pd.notna(x) else False
            )
        ]
        # 抽行(客户和专员各抽一个)，注意，如果有'/'，则分开抽
        if df_node_repeat.empty:
            continue
        sampled_row = df_node_repeat.sample(n=1).iloc[0]

        # 判断该行有没有继承，parent非零或者其他行parent的值有对应到该行，都是继承关系
        parent = sampled_row['parent(继承)']
        if pd.notna(parent):
            # 有继承，找到对应的行，加进去
            df_parent = df_node[df_node['uid'] == parent]
            if not df_parent.empty:
                # print(1)
                parent_row = df_parent.iloc[0]
                human_utterances_parent = parent_row['human(客户)'].split('/')
                assistant_utterances_parent = parent_row['assistant(专员)'].split('/')
                human_utterance_parent = np.random.choice(human_utterances_parent).strip()
                assistant_utterance_parent = np.random.choice(assistant_utterances_parent).strip()
                dialog.append([
                    {"role": "user", "content": human_utterance_parent}, 
                    {"role": "assistant", "content": assistant_utterance_parent}]
                )

            human_utterances = sampled_row['human(客户)'].split('/')
            assistant_utterances = sampled_row['assistant(专员)'].split('/')
            human_utterance = np.random.choice(human_utterances).strip()
            assistant_utterance = np.random.choice(assistant_utterances).strip()
            dialog.append([
                {"role": "user", "content": human_utterance},
                {"role": "assistant", "content": assistant_utterance}]
            )

            # 或者其他行parent的值有对应到该行，没有的话就不处理
            df_children = df_node[df_node['parent(继承)'] == sampled_row['uid']]
            if not df_children.empty:
                # print(2)
                # 抽取一行
                child_row = df_children.sample(n=1).iloc[0]
                human_utterances_child = child_row['human(客户)'].split('/')
                assistant_utterances_child = child_row['assistant(专员)'].split('/')
                human_utterance_child = np.random.choice(human_utterances_child).strip()
                assistant_utterance_child = np.random.choice(assistant_utterances_child).strip()
                dialog.append([
                    {"role": "user", "content": human_utterance_child},
                    {"role": "assistant", "content": assistant_utterance_child}]
                )
            
            # 将三个dialog_parent按顺序插入到dialog的assistant中，节点要求必须是资金困难、用卡及征信、放时间应对这三个节点其中一个，可以重复，也可以不插入;如果assistant本身有继承，那么不衔接施压话术 
            if node in insert_nodes and parent_index < len(dialog_parent):
                if df_children.empty and df_parent.empty:
                    if np.random.rand() < 0.7:
                        dialog[-1][1]['content'] += dialog_parent[parent_index]
                        parent_index += 1

    # 合并dialog
    dialog = [turn for pair in dialog for turn in pair]
    dialogs.append(dialog)
    
# print(dialogs)

In [11]:
len(dialogs)

40000

## 替换标签
- 案例标签
- <随机时间>：比<银行查账时间>大
- <随机金额>
- <empty_tag>
- <逾期金额1>：目前提取的第一笔，但这第一笔不一定是最少的那笔

In [ ]:
# 得到对话后，最后case信息替换标签
import os

case_path = 'cases_random'
cases = []
prompts = []
for case in os.listdir(case_path):
    case_file = os.path.join(case_path, case)
    
    data = {}
    # 读取case文件内容，txt格式，按行索引，存储为data
    with open(case_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    for line in lines:
        if line.startswith('- 专员工号'):
            data["专员工号"] = line.split('：')[1].strip()
        if line.startswith('- 姓名'):
            data["姓名"] = line.split('：')[1].strip()
        if line.startswith('- 性别'):
            data["性别"] = line.split('：')[1].strip()
            if data["性别"] == '男':
                data['性别'] = '先生'
            else:
                data['性别'] = '女士'
        if line.startswith('- 最大逾期天数'):
            data["最大逾期天数"] = line.split('：')[1].strip()
        if line.startswith('- 银行查账时间'):
            data["银行查账时间"] = line.split('：')[1].strip()
            # data['随机时间'] = 
        if line.startswith('- 逾期笔数'):
            data["逾期笔数"] = line.split('：')[1].strip()
        if line.startswith('- 逾期金额'):
            # 判断data["逾期笔数"]是不是1
            if data["逾期笔数"] == '1':
                data["逾期金额1"] = line.split('：')[1].strip()
            else:
                # data["逾期金额1"] = line.split('：')[1].strip().split('，')[0].strip()
                tmp = line.split('：')[1].strip().split('，')
                # 取最小的
                data["逾期金额1"] = min([x.strip() for x in tmp], key=lambda x: float(x))
        if line.startswith('- 剩余总待还'):
            data["剩余总待还"] = line.split('：')[1].strip()
            # data['随机金额'] = str(round(float(data["剩余总待还"]) * np.random.uniform(0.3, 0.7)))
    cases.append(data)
    prompts.append(''.join(lines))
    # print(data)

In [14]:
# 这里一一对应，和原本cases_random的顺序不一样
print(prompts[10])
print(cases[10])

项目：洋钱罐电催
- 专员工号：912631

- 姓名：王飞
- 性别：男
- 年龄：50+

- 账龄：S2
- 最大逾期天数：20
- 今天日期：2027-01-29
- 银行查账时间：今天中午12点

- 逾期笔数：4
- 逾期金额：2258.39，4661.61，2659.31，1216.50
- 剩余总待还：10795.81
{'专员工号': '912631', '姓名': '王飞', '性别': '先生', '最大逾期天数': '20', '银行查账时间': '今天中午12点', '逾期笔数': '4', '逾期金额1': '1216.50', '剩余总待还': '10795.81', '随机金额': '4878'}


In [15]:
# 遍历路径，依次代入case，case可重复
i=0
dialog_filled_all = []
for dialog in dialogs:
    # dialogs的数量比cases多，case按顺序重复使用
    case = cases[i]
    prompt = prompts[i]
    dialog_filled = []
    for turn in dialog:
        content = turn['content']
        content = content.replace('<专员工号>', case['专员工号'])
        content = content.replace('<姓名>', case['姓名'])
        content = content.replace('<性别>', case['性别'])
        content = content.replace('<最大逾期天数>', case['最大逾期天数'])
        content = content.replace('<银行查账时间>', case['银行查账时间'])
        # content = content.replace('<随机时间>', str(int(case['银行查账时间']) + np.random.randint(1, 5)))
        content = content.replace('<逾期笔数>', case['逾期笔数'])
        content = content.replace('<逾期金额1>', case['逾期金额1']+'元')
        content = content.replace('<剩余总待还>', case['剩余总待还']+'元')
        # content = content.replace('<随机金额>', case['随机金额'])
        content = content.replace('<empty_tag>', '')
        turn_filled = {
            "role": turn['role'],
            "content": content
        }
        dialog_filled.append(turn_filled)
    
    # dialog_filled 的第一个元素添加system
    dialog_filled.insert(0, {
        "role": "system",
        "content": "你是一个洋钱罐的催收专员，请根据客户的情况，使用合适的话术与客户进行沟通，争取让客户承诺还款。\n" + prompt
    })
    i = (i + 1) % len(cases)
    dialog_filled_all.append({"messages": dialog_filled})
    # print(dialog_filled)


In [16]:
len(dialog_filled_all)

40000

In [17]:
dialog_filled_all[100]

{'messages': [{'role': 'system',
   'content': '你是一个洋钱罐的催收专员，请根据客户的情况，使用合适的话术与客户进行沟通，争取让客户承诺还款。\n项目：洋钱罐电催\n- 专员工号：148652\n\n- 姓名：赵畅\n- 性别：男\n- 年龄：40+\n\n- 账龄：S2\n- 最大逾期天数：16\n- 今天日期：2026-11-16\n- 银行查账时间：今天下午1点\n\n- 逾期笔数：5\n- 逾期金额：2102.20，3350.13，1085.41，2647.02，2765.33\n- 剩余总待还：11950.09'},
  {'role': 'user', 'content': '你是哪个机构的？'},
  {'role': 'assistant', 'content': '我这边是洋钱罐的工作人员，你是赵畅先生对吧？'},
  {'role': 'user', 'content': '之前不是问过了吗，我是啊。'},
  {'role': 'assistant',
   'content': '你好，这边是洋钱罐贷后工作人员，本次通话全程录音，来电是关于你在平台办理的借款业务已经逾期16天了，逾期金额还有11950.09元没有处理，您是忘记还款了还是遇到什么困难了？'},
  {'role': 'user', 'content': '嗯。'},
  {'role': 'assistant', 'content': '那这边就帮你备注今天下午1点查账了，确定可以处理对吧？'},
  {'role': 'user', 'content': '我让朋友帮我还一下。'},
  {'role': 'assistant', 'content': '确定今天下午1点前可以处理进来是吧？'},
  {'role': 'user', 'content': '知道了。'},
  {'role': 'assistant',
   'content': '全程通话都是有录音的，既然答应了，就不要违约了，我们会准时查账的，不然后续可能就不是还钱这么简单的事情了。'}]}

In [18]:
# save to dialog_filled_all to json file
import json
with open('Yangqg-backbone-40000-1111.json', 'w', encoding='utf-8') as f:
    json.dump(dialog_filled_all, f, ensure_ascii=False, indent=4)